# Notebook 1: Prepare the data

**Goal of the workshop:** turn every customer in a CRM table into a **vector** (a list of 128 numbers), so we can

* search customers with plain English: *"unhappy healthcare customers"*
* find customers **similar** to a given one

```
data/crm.csv  →  [1. Prepare data]  →  [2. Train model]  →  [3. Search]
                   you are here
```

In this notebook we turn the CSV into numbers the model can read.

## Load the CSV

We read every column as text. We will convert the columns ourselves in the next steps.

In [ ]:
import pandas as pd

df = pd.read_csv("data/crm.csv", dtype = str, keep_default_na = False)
print(f"Shape: {df.shape}")

In [ ]:
df.head(3)

## Group the columns

A row mixes different kinds of data. Each kind gets its own treatment:

| Kind | Example | How we turn it into numbers |
|---|---|---|
| Category | `plan_tier = Enterprise` | give each value an id (1, 2, 3, …) |
| Number | `lifetime_value = 18004.87` | scale it to a similar range |
| Date | `signup_date = 2020-11-29` | convert it to *days since* |
| Text | `notes = "Frustrated by outages…"` | split into words, give each word an id |

`Names`, `emails` and `phone numbers` are **not** used. They identify a person but say nothing about behaviour.

In [ ]:
CAT_COLS = ["city", "state", "country", "timezone", "industry", "job_title", "plan_tier",
            "lifecycle_stage", "acquisition_source", "preferred_channel", "last_ticket_category",
            "newsletter_opt_in"]
NUM_COLS = ["total_orders", "monthly_recurring_revenue", "lifetime_value", "avg_order_value",
            "open_tickets", "satisfaction_score", "churn_risk"]
DATE_COLS = ["signup_date", "last_purchase_date", "last_contact_date"]
TEXT_COLS = ["notes", "interests", "last_ticket_subject"]

### A quick look

Customer Churn: The total number or percentage of individual customers who leave or fail to renew.

Unhappy customers have a higher churn risk, and their notes sound unhappy too. The columns tell the **same story**,
and that is what the model will learn from.

In [ ]:
satisfaction = pd.to_numeric(df.satisfaction_score)
churn = pd.to_numeric(df.churn_risk)

print("Unhappy customer:", df.notes[satisfaction <= 2].iloc[0])
print("Happy customer:  ", df.notes[satisfaction >= 9].iloc[0])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.5))
plt.scatter(satisfaction, churn, s=4, alpha=0.3)
plt.xlabel("Satisfaction Score")
plt.ylabel("Churn Risk")
plt.show()

## Keep some customers aside for testing

80% of customers are used for training and 20% for testing. We only learn statistics (averages, vocabularies) from the training part.

In [ ]:
import numpy as np

# Split the dataset into train and test sets
# The numbers are spread evenly between 0 and 1, so about 20% of them fall below 0.2
is_test = np.random.default_rng(0).random(len(df)) < 0.2
train = df[~is_test]  # keeps the training rows (~ flips True and False).
test = df[is_test]  # keeps the test rows.
print(f"Train:{len(train)}  Test: {len(test)}  Total: {len(df)}")

## Numbers and dates

* Dates become **days since** that date. "Signed up 2,000 days ago" is more useful to a model than a timestamp.
* Money and counts are very skewed (a few huge accounts), so we take `log(1 + x)`.
* Then every column is **standardized**: subtract the average, divide by the standard deviation.

In [ ]:
TODAY = pd.Timestamp("2026-10-01")  # "today" for this dataset
LOG_COLS = ["total_orders", "monthly_recurring_revenue", "lifetime_value", "avg_order_value",
            "days_since_signup", "days_since_last_purchase", "days_since_last_contact"]


def numeric_features(df):
    out = df[NUM_COLS].apply(pd.to_numeric, errors="coerce")
    for col in DATE_COLS:
        out["days_since_" + col.replace("_date", "")] = (TODAY - pd.to_datetime(df[col], errors="coerce")).dt.days
    out[LOG_COLS] = np.log1p(out[LOG_COLS])
    return out


num = numeric_features(df)
mean, std = num[~is_test].mean(), num[~is_test].std()
num.head()

Effect of Log

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
pd.to_numeric(df.lifetime_value).plot.hist(bins=50, ax=ax[0], title="lifetime_value")
num.lifetime_value.plot.hist(bins=50, ax=ax[1], title="log(1 + lifetime_value)")
plt.show()

## Categories

Each category column gets a small dictionary, `value → id`. Id **0** means "unknown" (a value never seen in training).

In [ ]:
vocab = {col: {value: i + 1 for i, value in enumerate(sorted(train[col].unique()))} for col in CAT_COLS}
vocab["plan_tier"]

## Text

We keep it simple and use no pretrained language model:

1. lowercase and split into words, then add word pairs (`billing_issue`)
2. turn each word into a number with a **hash function**, so no dictionary is needed

In [ ]:
import re, zlib

BUCKETS = 50_000  # number of possible word ids
TEXT_LEN = 100    # max words per text field


def text_to_ids(text, max_len=TEXT_LEN):
    words = re.findall(r"[a-z0-9]+", text.lower())
    words = words + [a + "_" + b for a, b in zip(words, words[1:])]
    # print(f"Words: {words}")
    ids = [1 + zlib.crc32(w.encode()) % (BUCKETS - 1) for w in words][:max_len]
    return ids + [0] * (max_len - len(ids))  # pad with 0


text_to_ids("Billing issue not resolved", max_len=10)

## Put it all together

`encode_table` turns any table of customers into four arrays:

| array | shape | content |
|---|---|---|
| `cat` | rows × 12 | category ids |
| `num` | rows × 10 | standardized numbers (NUM and DATE) |
| `text` | rows × 3 × 100 | word ids |
| `missing` | rows × 25 | `True` where a value is missing |

**Missing values** (no purchase date, no ticket, …) are not filled in with a guess. We mark them, and the model will ignore them.

```
print(f"Length of\nCAT_COLS: {len(CAT_COLS)}\nNUM_COLS: {len(NUM_COLS)}\nDATE_COLS: {len(DATE_COLS)}\nTEXT_COLS: {len(TEXT_COLS)}")
```

In [ ]:
stats = {"mean": mean.to_dict(), "std": std.to_dict(), "vocab": vocab}


def encode_table(df, stats):
    num = (numeric_features(df) - pd.Series(stats["mean"])) / pd.Series(stats["std"])
    cat = np.stack([df[c].map(stats["vocab"][c]).fillna(0) for c in CAT_COLS], axis=1).astype(int)
    text = np.array([[text_to_ids(t) for t in row] for row in df[TEXT_COLS].to_numpy()])
    missing = np.concatenate([
        cat == 0,                         # unknown category
        num.isna().to_numpy(),            # missing number or date
        df[TEXT_COLS].to_numpy() == "",   # empty text
    ], axis=1)
    return {"cat": cat, "num": num.fillna(0).to_numpy("float32"), "text": text, "missing": missing}


X = encode_table(df, stats)
for name, array in X.items():
    print(f"{name:8} {array.shape}")

In [ ]:
print("first customer, category ids:", X["cat"][0])
print("first customer, numbers:     ", X["num"][0].round(2))
print("first customer, missing:     ", X["missing"][0].astype(int))

## Tags: plain-English descriptions of each customer

To search with English, the model needs examples of *queries* and the customers they describe. We have none, so we
**make them up from the data**. Every customer gets a few **tags**, and each tag has some ways to say it:

```
"industry=Healthcare" → ["healthcare", "healthcare company"]
"unhappy"             → ["unhappy", "frustrated", "dissatisfied"]
```

In notebook 2 we combine random tags into queries such as *"frustrated healthcare company"*.

In [ ]:
STAGE_WORDS = {"active": ["active customer"], "at_risk": ["at risk customer", "at-risk account"],
               "churned": ["churned customer", "lost customer"], "lead": ["lead", "prospect"]}


def customer_tags(r):
    ind, plan, ticket = r.industry.lower(), r.plan_tier.lower(), r.last_ticket_category
    satisfaction = float(r.satisfaction_score)
    tags = {
        f"industry={r.industry}": [ind, f"{ind} company"],
        f"plan={r.plan_tier}": [f"{plan} plan", f"on {plan}"],
        f"city={r.city}": [f"in {r.city}", f"from {r.city}"],
        f"country={r.country}": [f"in {r.country}", f"from {r.country}"],
        f"stage={r.lifecycle_stage}": STAGE_WORDS[r.lifecycle_stage],
    }
    if ticket:
        tags[f"ticket={ticket}"] = [f"{ticket} issue", f"{ticket} problem"]

    if satisfaction >= 8:
        tags["happy"] = ["happy", "satisfied", "very happy customer"]

    if satisfaction < 5:
        tags["unhappy"] = ["unhappy", "frustrated", "dissatisfied"]

    if float(r.churn_risk) > 0.55:
        tags["churn_risk"] = ["likely to churn", "high churn risk"]

    if float(r.lifetime_value) > 20000:
        tags["high_value"] = ["high value", "big spender", "top revenue"]

    for topic in r.interests.split("; "):
        tags[f"interest={topic}"] = [topic, f"interested in {topic}"]
    return tags


tags = [customer_tags(r) for r in df.itertuples()]
tags[0]

## Save

Notebook 2 continues from these files.

In [ ]:
import json
from pathlib import Path

out = Path("outputs/prepared")
out.mkdir(parents=True, exist_ok=True)

np.savez(out / "data.npz", **X, is_test=is_test)
(out / "stats.json").write_text(json.dumps(stats))
(out / "tags.json").write_text(json.dumps(tags))
print("saved:", [p.name for p in out.iterdir()])

## Recap

* Every column became numbers: **ids** for categories and words, **scaled values** for numbers and dates.
* Missing values are **marked**, not guessed.
* Every customer has **tags** that we will use to create practice search queries.

**Next:** `02_model_and_training.ipynb`